# I will now be building the Deiml-based transformation operations from the Clifford + T gate set

Allowed gates are Clifford gates: H, S, X, Y, Z, I, CNOT and all of their inverses  (add to this list any other Clifford gate I need, this isn't meant to be exhaustive)

and the only non-Clifford gate allowed is T

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Operator, Statevector
import numpy as np
from matplotlib import pyplot as plt

from Elementary_circuits import subarithmetic_circuits
from Elementary_circuits import basic_arithmetic_circuits
from Elementary_circuits import midsize_basic_circuits
from Elementary_circuits import transformation_circuits

np.set_printoptions(legacy='1.25') # prevent the type being printed in numpy output

## Test $M_2^*$

For every computational basis input, post-select `ps` in $|0\rangle$ and verify that the resulting block has support only on $|2x-1 \bmod 2^{L_f+2}\rangle$ and $|2x \bmod 2^{L_f+2}\rangle$. The two Hadamards give each output an unnormalized amplitude of $1/2$.

In [2]:
L_f = 2
n = L_f + 2
modulus = 1 << n
ancilla_count = max(0, n - 3)

for input_value in range(modulus):
    x = QuantumRegister(n, 'x')
    ps = QuantumRegister(1, 'ps')
    ancillas = QuantumRegister(ancilla_count, 'ancilla')
    qc = QuantumCircuit(x, ps, ancillas)

    for bit in range(n):
        if (input_value >> bit) & 1:
            qc.x(x[bit])

    transformation_circuits.M_2_star(qc, L_f, x, ps, ancillas)
    state = Statevector.from_instruction(qc)

    selected_amplitudes = np.array([
        state.data[sum(((output >> bit) & 1) << qc.find_bit(x[bit]).index
                       for bit in range(n))]
        for output in range(modulus)
    ])
    expected = np.zeros(modulus, dtype=complex)
    expected[(2 * input_value - 1) % modulus] = 0.5
    expected[(2 * input_value) % modulus] = 0.5

    np.testing.assert_allclose(selected_amplitudes, expected, atol=1e-12)

    ancilla_mask = sum(1 << qc.find_bit(qubit).index for qubit in ancillas)
    assert all(abs(amplitude) < 1e-12
               for index, amplitude in enumerate(state.data)
               if index & ancilla_mask)

print(f'M_2_star passed for all {modulus} basis states.')

M_2_star passed for all 16 basis states.
